# Case 2: reproducing the official triage numbers

This notebook recomputes every estimate of the body-variant triage from the published extract `data/body_triage_lives.parquet` and compares it, number by number, with `data/body_triage_reference.json` (the values computed in the original study). Nothing new is analysed here.

**The triage in one paragraph.** 480 simulated lives of 14 days of the current version of a virtual pet: 2 *arms* (`control`, the pet as it is; `body_variant`, the same pet with a variant of its body physiology) × 12 *cells* (4 owner types × 3 places of the food) × 20 *units*. A unit is paired across arms: the same (unit, owner, place) appears once in each arm. The triage only estimates the size of the effect: it has no formal reading. See `data/SCHEMA.md` for the columns.

## Setup
DuckDB reads the parquet; `scipy` only supplies the exact Student-t quantile.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats

from casestudy import triage as tr

REF = tr.REF
TOL = 1e-9
print(REF["lives"], "lives,", REF["units"], "units; formal reading:", REF["formal_reading"])

480 lives, 20 units; formal reading: False


## Step 1: the data (gate E1)
480 lives, 240 per arm, 20 per arm and cell, and complete pairing: every (unit, owner, place) appears in both arms. The arm means of `time_well` must equal the official levels.

In [2]:
counts = tr.sql('''
    SELECT arm, count(*) AS lives, count(DISTINCT owner || '|' || place) AS cells,
           count(DISTINCT unit) AS units, min(n) AS min_per_cell, max(n) AS max_per_cell
    FROM (SELECT *, count(*) OVER (PARTITION BY arm, owner, place) AS n FROM triage)
    GROUP BY arm ORDER BY arm''')
pairing = tr.sql('''
    SELECT count(*) AS unit_cells, min(k) AS min_arms, max(k) AS max_arms
    FROM (SELECT unit, owner, place, count(DISTINCT arm) AS k FROM triage GROUP BY ALL)''')
means = tr.sql("SELECT arm, avg(time_well) AS time_well FROM triage GROUP BY arm ORDER BY arm")
assert counts.lives.sum() == REF["lives"] == 480
assert (counts.lives == 240).all() and (counts.cells == 12).all() and (counts.units == 20).all()
assert (counts.min_per_cell == 20).all() and (counts.max_per_cell == 20).all()
assert pairing.unit_cells[0] == 240 and pairing.min_arms[0] == pairing.max_arms[0] == 2
e1 = {r.arm: abs(r.time_well - REF["arm_means"][r.arm]["time_well"]) for r in means.itertuples()}
assert max(e1.values()) <= TOL, e1
assert {r.arm: round(r.time_well, 3) for r in means.itertuples()} == {"body_variant": 0.752, "control": 0.728}
display(counts, pairing, means)
print(f"E1 PASSED: largest |diff| of the time_well arm means = {max(e1.values()):.2e}")

,arm,lives,cells,units,min_per_cell,max_per_cell
0,body_variant,240,12,20,20,20
1,control,240,12,20,20,20


,unit_cells,min_arms,max_arms
0,240,2,2


,arm,time_well
0,body_variant,0.751736
1,control,0.727948


E1 PASSED: largest |diff| of the time_well arm means = 6.66e-16


## Step 2: the method
For a measure and a set of cells, the official method works unit by unit: in each of the unit's cells take `body_variant − control`, then average those differences over the cells of the set. That gives one number per unit, so each estimate rests on 20 paired differences. The estimate is their mean; the 95% interval is mean ± t × sd / √20, with sd the sample standard deviation (ddof = 1) and t the Student-t quantile with 19 degrees of freedom. The arm levels are averaged the same way (per unit over the cells, then over units).

The sets of cells:

- **12 cells:** all of them.
- **11 cells:** without (`inconsistent`, `at_feeder`). In every life that cell is identical to (`attentive`, `at_feeder`) (Step 5), so the plan also reports the estimate over the 11 distinct cells.
- **by place:** the 4 cells of one place.
- **neglectful and random:** the 6 cells of the two owner types that let the food run out.
- **one cell:** a single (owner, place).

In [3]:
t19 = stats.t.ppf(0.975, 19)
d = tr.per_unit("time_well", "12_cells")
print(f"t(0.975, 19) = {t19:.6f}; first three per-unit differences:", d["diff"].head(3).round(5).tolist())

t(0.975, 19) = 2.093024; first three per-unit differences: [0.02207, 0.06523, 0.01823]


## Step 3: every official estimate
The official result lists 14 estimates: time well over 12 cells, 11 cells and each place; health over 12 cells, over the 6 neglectful/random cells and in each of those 6 cells; and the episode rate in `far_fixed`. Each is recomputed here (levels, mean, sd, interval) and compared with the reference.

In [4]:
rows = []
for name, ref in REF["estimates"].items():
    measure, group = tr.split(name)
    got = tr.estimate(measure, group)
    rows.append(dict(measure=measure, group=group, control=got["control"], body_variant=got["body_variant"],
                     estimate=got["mean"], ci_low=got["ci_low"], ci_high=got["ci_high"], sd=got["sd"],
                     n=got["n"], max_abs_diff=tr.max_diff(got, ref)))
est = pd.DataFrame(rows)
sci = lambda df: df.round(4).assign(max_abs_diff=df.max_abs_diff.map("{:.1e}".format))
sci(est)

,measure,group,control,body_variant,estimate,ci_low,ci_high,sd,n,max_abs_diff
0,health_d8_14,12_cells,0.6549,0.6909,0.0360,0.0052,0.0668,0.0658,20,1.1e-16
1,health_d8_14,neglectful_and_random,0.4794,0.5014,0.0220,-0.0043,0.0483,0.0561,20,2.1e-17
2,health_d8_14,random|at_feeder,0.3334,0.3035,-0.0299,-0.0472,-0.0126,0.0370,20,5.6e-17
3,health_d8_14,random|far,0.1477,0.1463,-0.0014,-0.0554,0.0526,0.1153,20,2.1e-17
4,health_d8_14,random|far_fixed,0.2058,0.2463,0.0405,-0.0243,0.1053,0.1385,20,1.4e-17
5,health_d8_14,neglectful|at_feeder,0.9710,0.9602,-0.0108,-0.0161,-0.0055,0.0114,20,3.5e-18
6,health_d8_14,neglectful|far,0.4303,0.5205,0.0901,-0.0446,0.2249,0.2879,20,4.2e-17
7,health_d8_14,neglectful|far_fixed,0.7881,0.8317,0.0436,-0.0343,0.1214,0.1664,20,2.2e-16
8,episode_rate,far_fixed,0.5281,0.6409,0.1128,0.0624,0.1632,0.1077,20,1.1e-16
9,time_well,11_cells,0.7048,0.7297,0.0249,0.0092,0.0406,0.0336,20,6.9e-18


In [5]:
for name in ("time_well|12_cells", "time_well|11_cells"):
    got = est[(est.measure + "|" + est.group) == name].estimate.iloc[0] / tr.MARGIN
    assert abs(got - REF["effect_over_margin"][name]) <= TOL
headline = {"time_well|12_cells": "+0.024 [+0.009; +0.038]", "time_well|11_cells": "+0.025 [+0.009; +0.041]"}
fmt = lambda r: f"{r.estimate:+.3f} [{r.ci_low:+.3f}; {r.ci_high:+.3f}]"
check = [dict(estimate=k, reproduced=fmt(est[(est.measure + "|" + est.group) == k].iloc[0]), official=v)
         for k, v in headline.items()]
pd.DataFrame(check)

,estimate,reproduced,official
0,time_well|12_cells,+0.024 [+0.009; +0.038],+0.024 [+0.009; +0.038]
1,time_well|11_cells,+0.025 [+0.009; +0.041],+0.025 [+0.009; +0.041]


## Step 4: the descriptive tables
The official result also reports, without any test, every published measure by set of cells (12, 11 and each place), the share of lives with time well above 0.99, and time well in each of the 12 cells. All recomputed and compared.

In [6]:
NEAR = f"CAST(time_well > {tr.NEAR_CEILING} AS DOUBLE)"
desc = []
for group, by in REF["by_group"].items():
    for measure, ref in by.items():
        expr = NEAR if measure == "share_time_well_above_0_99" else measure
        got = tr.estimate(expr, group)
        desc.append(dict(group=group, measure=measure, control=got["control"], body_variant=got["body_variant"],
                         estimate=got["mean"], ci_low=got["ci_low"], ci_high=got["ci_high"],
                         max_abs_diff=tr.max_diff(got, ref)))
desc = pd.DataFrame(desc)
cells = []
for c, ref in REF["time_well_by_cell"].items():
    got = tr.estimate("time_well", c)
    cells.append(dict(cell=c, control=got["control"], body_variant=got["body_variant"], estimate=got["mean"],
                      ci_low=got["ci_low"], ci_high=got["ci_high"], max_abs_diff=tr.max_diff(got, ref)))
cells = pd.DataFrame(cells)
print(len(desc), "descriptive rows;", len(cells), "cells")
display(sci(desc[desc.group == "12_cells"]), sci(cells))

55 descriptive rows; 12 cells


,group,measure,control,body_variant,estimate,ci_low,ci_high,max_abs_diff
0,12_cells,time_well,0.7279,0.7517,0.0238,0.0095,0.0381,1.1e-16
1,12_cells,time_well_d8_14,0.6658,0.7018,0.0360,0.0070,0.0649,1.1e-16
2,12_cells,health_d8_14,0.6549,0.6909,0.0360,0.0052,0.0668,1.1e-16
3,12_cells,deprivation,0.2602,0.2395,-0.0207,-0.0349,-0.0065,1.0e-17
4,12_cells,exhaustion,0.1149,0.0689,-0.0460,-0.0536,-0.0383,2.8e-17
5,12_cells,episode_rate,0.5471,0.6513,0.1042,0.0809,0.1274,6.9e-18
6,12_cells,meals_per_day,3.1235,3.6274,0.5039,0.4258,0.5819,1.1e-16
7,12_cells,sleep_frac_ticks,0.4647,0.3800,-0.0847,-0.0868,-0.0826,1.1e-16
8,12_cells,kicks,22.7542,19.9250,-2.8292,-3.7702,-1.8881,3.6e-15
9,12_cells,play_decisions,69.7750,77.8042,8.0292,5.8136,10.2448,1.4e-14


,cell,control,body_variant,estimate,ci_low,ci_high,max_abs_diff
0,random|at_feeder,0.5288,0.5133,-0.0156,-0.0234,-0.0077,1.1e-16
1,random|far,0.3804,0.4017,0.0214,-0.0063,0.0491,1.1e-16
2,random|far_fixed,0.4265,0.4476,0.0211,-0.0010,0.0432,1.1e-16
3,attentive|at_feeder,0.9828,0.9943,0.0115,0.0030,0.0200,2.2e-16
4,attentive|far,0.7306,0.7646,0.0340,-0.0560,0.1240,2.2e-16
5,attentive|far_fixed,0.8325,0.9374,0.1049,0.0331,0.1767,2.2e-16
6,inconsistent|at_feeder,0.9828,0.9943,0.0115,0.0030,0.0200,2.2e-16
7,inconsistent|far,0.7805,0.8260,0.0456,-0.0215,0.1126,2.2e-16
8,inconsistent|far_fixed,0.9436,0.9704,0.0269,-0.0378,0.0915,2.2e-16
9,neglectful|at_feeder,0.8795,0.8520,-0.0275,-0.0334,-0.0217,6.9e-18


## Step 5: why 11 distinct cells
The official result reports that (`inconsistent`, `at_feeder`) came out identical to (`attentive`, `at_feeder`) in all 20 lives of each arm (it compares the full event logs). The published measures show the same: in `at_feeder` every measure is equal for the two owners in every unit; in the other two places the two owners differ in every unit.

In [7]:
cols = " AND ".join(f"a.{m} = i.{m}" for m in tr.MEASURES)
same = tr.sql(f'''
    SELECT a.arm, a.place, count(*) AS lives, CAST(sum(CAST(({cols}) AS INT)) AS INT) AS equal
    FROM triage a JOIN triage i USING (arm, place, unit)
    WHERE a.owner = 'attentive' AND i.owner = 'inconsistent'
    GROUP BY ALL ORDER BY a.arm, a.place''')
for r in same.itertuples():
    ref = REF["inconsistent_equals_attentive"][f"{r.arm}|{r.place}"]
    assert (r.lives, r.equal) == (ref["lives"], ref["equal"]), (r, ref)
assert same[same.place == "at_feeder"].equal.eq(20).all() and same[same.place != "at_feeder"].equal.eq(0).all()
same

,arm,place,lives,equal
0,body_variant,at_feeder,20,20
1,body_variant,far,20,0
2,body_variant,far_fixed,20,0
3,control,at_feeder,20,20
4,control,far,20,0
5,control,far_fixed,20,0


## Gate E2
Every estimate, every descriptive row and every cell must be within 1e-9 of the reference, with n = 20. If anything fails, this cell stops the notebook; no tolerance or method is adjusted to make it pass.

In [8]:
assert len(est) == len(REF["estimates"]) == 14 and (est.n == 20).all()
assert len(desc) == 5 * 11 and len(cells) == 12
worst = max(est.max_abs_diff.max(), desc.max_abs_diff.max(), cells.max_abs_diff.max())
bad = pd.concat([est[est.max_abs_diff > TOL], desc[desc.max_abs_diff > TOL], cells[cells.max_abs_diff > TOL]])
assert bad.empty, "Mismatch vs reference:\n" + bad.to_string()
assert all(c["reproduced"] == c["official"] for c in check)
print(f"GATE E2 PASSED: 14/14 estimates, {len(desc)} descriptive rows and 12 cells within {TOL:g} "
      f"of the reference (largest |diff| = {worst:.2e}); the two headline numbers match.")

GATE E2 PASSED: 14/14 estimates, 55 descriptive rows and 12 cells within 1e-09 of the reference (largest |diff| = 1.42e-14); the two headline numbers match.
